# TweetAge v9: LSTM de caracteres como modelo de lenguaje por clase

Seis modelos generativos entrenados desde cero estiman log P(tuit | edad); se clasifica por suma de log-probabilidades de caracteres, incluido EOS, mas log prior. No se divide por longitud ni se calibran sesgos. Entrada one-hot mediante una matriz identidad fija, no una representacion aprendida ni preentrenada.

**Parametros fijados:** semilla 42; LSTM de una capa, hidden=256; dropout de salida 0.2; Adam lr=0.001; batch=128; maximo 30 epocas; paciencia=3; 10 % interno de grupos del entrenamiento para early stopping. PyTorch no aplica dropout recurrente entre capas cuando solo hay una: se usa nn.Dropout(0.2) sobre las salidas de la LSTM. No se afina ningun parametro con validacion exterior.

Se reutiliza **split_idx_v6_seed42.npz**, con hash del train y limpieza verificados. Se reconstruyen los cinco pliegues agrupados de v6 y se recalcula KN (caracteres 4, palabras 3, lambda=0.75, sin calibracion) en esos mismos pliegues. Se avanza solo si la media de las diferencias F1(v9)-F1(v6) es positiva y mayor que su desviacion muestral, ddof=1. El holdout se usa una sola vez si pasa; en siguientes ejecuciones solo se recupera su resultado guardado.

**Diferencias respecto al GRU localizado en el repo:** aquel usa v7_lower, 280 caracteres, GRU de 128 unidades y mezcla con KN de palabras. Esta v9 sigue el pedido: particion de v6, 140 caracteres y LSTM puro por clase. Comparten el concepto one-hot, pero los scores publicados de GRU no constituyen una comparacion directa. v6 mantiene grupos sensibles a mayusculas; se informa tambien el solapamiento en minusculas, sin cambiar la particion autorizada.

**Google Colab / Drive:** entradas en `/content/drive/MyDrive/PLN/tweet-age-2026-puj/`: `age_train.csv`, `age_test.csv`, `age_submission.csv` y `split_idx_v6_seed42.npz`. El NPZ no esta actualmente en esa carpeta segun la configuracion comunicada: debe subirse antes de ejecutar. Si falta, el notebook se detiene sin crear otra particion. No se usa carga manual con files.upload.

La submission condicional se escribe en `/content/drive/MyDrive/PLN/RESULTS/tweetage_v9_submission.csv`. Checkpoints, OOF y metricas se guardan en `/content/drive/MyDrive/PLN/RESULTS/tweetage_v9/<RUN_ID>/` para reanudar. El score publico 0.45737 de v6 es solo contexto aportado por el usuario y no se compara directamente con holdout.

Primero ejecute las celdas hasta la verificacion de normalizacion y pliegues, sin iniciar entrenamiento. Cuando los cuatro archivos, esquemas y hashes sean correctos, active GPU en Colab y ejecute **Restart & Run All**. Si la sesion se corta, vuelva a ejecutar desde el inicio para recuperar los checkpoints de Drive. No salte celdas ni cambie parametros para eludir un error.

No se instalan paquetes, no se modifica v6 ni se envia nada a Kaggle. Si cambia codigo o parametros, cambie RUN_ID para no reutilizar caches incompatibles. Las discrepancias de esquema, hash o cache detienen la ejecucion con instrucciones explicitas.

## Configuracion y almacenamiento
Se fijan semillas y parametros antes de leer validacion. Las entradas tienen nombres explicitos en la carpeta PLN de Drive; UPLOAD_INPUTS permanece en False. Esta celda no carga datos ni entrena. Los artefactos se separan de la submission final.

In [1]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import ast
import csv
import gc
import hashlib
import html
import io
import json
import math
import platform
import random
import re
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

UPLOAD_INPUTS=False
DATA_DIR=Path('/content/drive/MyDrive/PLN/tweet-age-2026-puj')
ARTIFACT_DIR=Path('/content/drive/MyDrive/PLN/RESULTS/tweetage_v9')
SUBMISSION_PATH=Path('/content/drive/MyDrive/PLN/RESULTS/tweetage_v9_submission.csv')
RUN_ID='v9_lstm_onehot_h256_c140_seed42_v1'
TRAIN_FILENAME='age_train.csv'
TEST_FILENAME='age_test.csv'
SAMPLE_FILENAME='age_submission.csv'
SPLIT_FILENAME='split_idx_v6_seed42.npz'
CONFIRMED_COLUMNS={'train':None,'test':None,'sample':None}
N_PARALELO=6
DETERMINISTA=True
PRECISION='fp32'
CPU_THREADS_PER_JOB=1
BENCHMARK_ID='a100_fp32_seed42_v1'
SEED=42
PARAMS=dict(hidden=256,layers=1,dropout=0.2,lr=1e-3,batch=128,max_epochs=30,patience=3,inner_fraction=0.1,max_chars=140)
CLASSES=np.array(['13-17','18-24','25-34','35-49','50-64','65-xx'])
PAD_ID,UNK_ID,BOS_ID,EOS_ID=0,1,2,3
BASE_EPSILON=1e-6
BOS,EOS,UNK='\x02','\x03','\x01'
IMPLEMENTATION_VERSION='v9_lstm_lm_onehot_20261016_v1'
assert PRECISION=='fp32','Esta version no autoriza fp16, bf16 ni TF32.'
assert isinstance(N_PARALELO,int) and N_PARALELO>=1

def seed_everything(seed=SEED):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=DETERMINISTA
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.allow_tf32=False
    torch.backends.cuda.matmul.allow_tf32=False
    torch.use_deterministic_algorithms(DETERMINISTA)

seed_everything()
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python',platform.python_version(),'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__,'torch',torch.__version__)
print('Dispositivo',DEVICE,'| semillas random/numpy/torch/cuda:',SEED)
print('Parametros fijos:',PARAMS)
print('Ejecucion: N_PARALELO=',N_PARALELO,'DETERMINISTA=',DETERMINISTA,'PRECISION=',PRECISION,'TF32=False')
print('Cada trabajo conserva semilla privada 42 para coincidir con la inicializacion secuencial original.')
if DEVICE.type!='cuda':print('Aviso: las pruebas reales de rendimiento y CV requieren GPU; no se reduce el experimento silenciosamente.')
print('Datos configurados:',DATA_DIR)
print('Artefactos configurados:',ARTIFACT_DIR/RUN_ID)
print('Submission configurada:',SUBMISSION_PATH)

Python 3.13.15 numpy 2.1.3 pandas 2.2.3 sklearn 1.6.1 torch 2.11.0+cu130
Dispositivo cuda | semillas random/numpy/torch/cuda: 42
Parametros fijos: {'hidden': 256, 'layers': 1, 'dropout': 0.2, 'lr': 0.001, 'batch': 128, 'max_epochs': 30, 'patience': 3, 'inner_fraction': 0.1, 'max_chars': 140}
Ejecucion: N_PARALELO= 6 DETERMINISTA= True PRECISION= fp32 TF32=False
Cada trabajo conserva semilla privada 42 para coincidir con la inicializacion secuencial original.
Datos configurados: /content/drive/MyDrive/PLN/tweet-age-2026-puj
Artefactos configurados: /content/drive/MyDrive/PLN/RESULTS/tweetage_v9/v9_lstm_onehot_h256_c140_seed42_v1
Submission configurada: /content/drive/MyDrive/PLN/RESULTS/tweetage_v9_submission.csv


## Verificacion de Drive antes de cargar datos
Monte Drive y compruebe las cuatro entradas configuradas. RESULTS se crea si hace falta. Si falta `split_idx_v6_seed42.npz`, subalo a DATA_DIR desde los artefactos originales de v6: no se genera otra particion y no se continua con la carga ni con el entrenamiento.

In [2]:
from google.colab import drive

def verify_drive_inputs():
    if UPLOAD_INPUTS:
        raise ValueError('UPLOAD_INPUTS debe ser False: esta version usa los archivos ya presentes en Google Drive.')
    required={
        'train':DATA_DIR/TRAIN_FILENAME,
        'test':DATA_DIR/TEST_FILENAME,
        'sample':DATA_DIR/SAMPLE_FILENAME,
        'split':DATA_DIR/SPLIT_FILENAME,
    }
    SUBMISSION_PATH.parent.mkdir(parents=True,exist_ok=True)
    missing=[path for path in required.values() if not path.is_file()]
    print('Directorio RESULTS verificado:',SUBMISSION_PATH.parent)
    for label,path in required.items():
        print(label,':',path,'OK' if path.is_file() else 'FALTA')
    if missing:
        details='\n'.join(str(path) for path in missing)
        raise FileNotFoundError(
            'Verificacion previa fallida. Faltan archivos de entrada:\n'+details+
            '\nSuba los archivos que faltan a DATA_DIR antes de continuar. '+
            'En particular, split_idx_v6_seed42.npz debe copiarse desde v6 a esa carpeta. '+
            'No se genera ni se sustituye ninguna particion; no se cargan datos ni se entrena.'
        )
    return required

drive.mount('/content/drive')
INPUT_PATHS=verify_drive_inputs()
TRAIN_PATH=INPUT_PATHS['train']
TEST_PATH=INPUT_PATHS['test']
SAMPLE_PATH=INPUT_PATHS['sample']
SPLIT_PATH=INPUT_PATHS['split']
RUN_DIR=ARTIFACT_DIR/RUN_ID
RUN_DIR.mkdir(parents=True,exist_ok=True)
print('Cuatro entradas presentes. Artefactos:',RUN_DIR)
print('Submission final:',SUBMISSION_PATH)

Mounted at /content/drive
Directorio RESULTS verificado: /content/drive/MyDrive/PLN/RESULTS
train : /content/drive/MyDrive/PLN/tweet-age-2026-puj/age_train.csv OK
test : /content/drive/MyDrive/PLN/tweet-age-2026-puj/age_test.csv OK
sample : /content/drive/MyDrive/PLN/tweet-age-2026-puj/age_submission.csv OK
split : /content/drive/MyDrive/PLN/tweet-age-2026-puj/split_idx_v6_seed42.npz OK
Cuatro entradas presentes. Artefactos: /content/drive/MyDrive/PLN/RESULTS/tweetage_v9/v9_lstm_onehot_h256_c140_seed42_v1
Submission final: /content/drive/MyDrive/PLN/RESULTS/tweetage_v9_submission.csv


## Lectura y esquema de los CSV confirmados
Se leen unicamente los cuatro archivos indicados en la configuracion, despues de la verificacion de Drive. Train y test requieren punto y coma con BOM; la plantilla `age_submission.csv`, coma sin BOM. Se imprimen columnas, tipos, filas, distribucion y ejemplos. Los esquemas deben ser exactamente id/text/age_range, id/text e id/age_range, respectivamente; si difieren, se muestran y se detiene la ejecucion.

In [3]:
def read_raw(path,label):
    with path.open('rb') as handle:prefix=handle.read(32768)
    first=prefix.decode('utf-8-sig').splitlines()[0]
    separator=csv.Sniffer().sniff(first,delimiters=',;\t|').delimiter
    has_bom=prefix.startswith(b'\xef\xbb\xbf')
    expected_separator=';' if label in {'train','test'} else ','
    expected_bom=label in {'train','test'}
    if separator!=expected_separator or has_bom!=expected_bom:
        raise ValueError(f'Formato inesperado en {path.name}: separador={separator!r}, BOM={has_bom}. Esperado: separador={expected_separator!r}, BOM={expected_bom}. No se continua.')
    frame=pd.read_csv(path,sep=separator,encoding='utf-8-sig',keep_default_na=False,dtype=str)
    print(label,'archivo',path.name,'separador',repr(separator),'BOM',has_bom,'filas',len(frame))
    print('Columnas detectadas:',frame.columns.tolist())
    print('Tipos leidos (strings para conservar IDs y textos literales):',frame.dtypes.astype(str).to_dict())
    display(frame.head(3))
    if label=='train' and 'age_range' in frame.columns:
        display(frame['age_range'].value_counts(dropna=False).rename('filas').to_frame())
    return frame

def confirm_schema(frame,label,required):
    actual=frame.columns.tolist()
    print(label,'columnas esperadas:',required,'| detectadas:',actual)
    if actual!=required:
        raise ValueError(f'Esquema inesperado en {label}. Columnas detectadas: {actual}; esperadas: {required}. Detener y confirmar los archivos antes de continuar.')
    mapping={name:name for name in required}
    return frame[required].copy(),mapping

assert all(path.is_file() for path in INPUT_PATHS.values()),'Una entrada dejo de estar disponible; ejecute la verificacion de Drive antes de cargar.'
train_raw=read_raw(TRAIN_PATH,'train')
train_raw,train_mapping=confirm_schema(train_raw,'train',['id','text','age_range'])
test_raw=read_raw(TEST_PATH,'test')
test_raw,test_mapping=confirm_schema(test_raw,'test',['id','text'])
sample_raw=read_raw(SAMPLE_PATH,'sample')
sample,sample_mapping=confirm_schema(sample_raw,'sample',['id','age_range'])
assert set(train_raw.age_range)==set(CLASSES),'Los seis rangos esperados no coinciden; confirme las etiquetas antes de continuar.'
assert train_raw.id.is_unique and test_raw.id.is_unique and sample.id.is_unique
print('Filas train/test/plantilla:',len(train_raw),len(test_raw),len(sample_raw))
print('Distribucion de rangos confirmada:')
display(train_raw.age_range.value_counts().reindex(CLASSES).rename('filas').to_frame())
TRAIN_SHA=hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest()
SPLIT_SHA=hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest()
print('SHA-256 train:',TRAIN_SHA)
print('SHA-256 NPZ de v6:',SPLIT_SHA)

train archivo age_train.csv separador ';' BOM True filas 169336
Columnas detectadas: ['id', 'text', 'age_range']
Tipos leidos (strings para conservar IDs y textos literales): {'id': 'object', 'text': 'object', 'age_range': 'object'}


,id,text,age_range
0,1,Ud y su flia fueron las mas beneficiadas con O...,50-64
1,2,http://t.co/Jp4hFcwMF6 ...,13-17
2,3,Hasta el 1 de oct Medellin estara respirando p...,35-49


,filas
age_range,
13-17,28863
18-24,28676
25-34,28347
35-49,28230
50-64,28060
65-xx,27160


train columnas esperadas: ['id', 'text', 'age_range'] | detectadas: ['id', 'text', 'age_range']
test archivo age_test.csv separador ';' BOM True filas 112892
Columnas detectadas: ['id', 'text']
Tipos leidos (strings para conservar IDs y textos literales): {'id': 'object', 'text': 'object'}


,id,text
0,1,RT @diana_dagostino: La tragedia de Choroni en...
1,2,RT @SignosDelRock: Ojala vos tambien. https://...
2,3,"@prioritycaro A mi tambien me parece eso, pero..."


test columnas esperadas: ['id', 'text'] | detectadas: ['id', 'text']
sample archivo age_submission.csv separador ',' BOM False filas 112892
Columnas detectadas: ['id', 'age_range']
Tipos leidos (strings para conservar IDs y textos literales): {'id': 'object', 'age_range': 'object'}


,id,age_range
0,1,13-17
1,2,13-17
2,3,13-17


sample columnas esperadas: ['id', 'age_range'] | detectadas: ['id', 'age_range']
Filas train/test/plantilla: 169336 112892 112892
Distribucion de rangos confirmada:


,filas
age_range,
13-17,28863
18-24,28676
25-34,28347
35-49,28230
50-64,28060
65-xx,27160


SHA-256 train: 39c6c8270366fe155f3c8dd232e31867bee659af8908121d04b6bd89bd163748
SHA-256 NPZ de v6: fb299a3ef97a2a6948685163ba80aef5705c442e9ccda66c2c1fb0bce1e43c4d


## Normalizacion y particiones de v6
Las funciones URL/menciones/repeticiones se copian literalmente de v6. Se usa su limpieza previa y se comprueba el hash y las filas retenidas del NPZ: asi no se cambia la particion por diferencias de orden de operaciones. Se decodifica HTML, se normaliza y se excluyen vacios/artefactos; no se eliminan mayusculas, emojis ni puntuacion salvo lo comprendido por la regla URL original. El test no influye en vocabularios, entrenamiento o seleccion.

In [4]:
URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)

MENTION_RE = re.compile(r'@\w+', re.UNICODE)

REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

clean_text=train_raw.text.fillna('').str.strip().map(html.unescape)
excluded=clean_text.eq('')|clean_text.str.contains('#\u00bfNOMBRE',regex=False)
train=train_raw.loc[~excluded].copy()
train['raw_row']=np.flatnonzero(~excluded.to_numpy())
train['text']=clean_text.loc[~excluded]
train=train.reset_index(drop=True)
train['normalized']=train.text.map(normalize_text)
post_mask=train_raw.text.fillna('').str.strip().map(html.unescape).map(normalize_text)
post_excluded=post_mask.eq('')|post_mask.str.contains('#\u00bfNOMBRE',regex=False)
assert np.array_equal(excluded.to_numpy(),post_excluded.to_numpy()),'El orden de exclusion propuesto cambia filas respecto a v6; detener y confirmar antes de crear otra particion.'
y_all=pd.Categorical(train.age_range,categories=CLASSES).codes.astype(np.int64)
with np.load(SPLIT_PATH,allow_pickle=False) as saved:split={name:saved[name].copy() for name in saved.files}
assert str(split['train_sha256'])==TRAIN_SHA,'El train no coincide byte a byte con el de v6; no reutilizar indices ni crear una particion nueva.'
assert int(split['seed'])==SEED
assert np.array_equal(split['retained_raw_idx'],train.raw_row.to_numpy())
group_ids,unique_texts=pd.factorize(train.normalized,sort=True)
assert np.array_equal(group_ids,split['group_ids'])
idx_development,idx_holdout=split['train_idx'],split['valid_idx']
dev_groups,holdout_groups=split['train_groups'],split['valid_groups']
assert np.array_equal(np.sort(np.concatenate([idx_development,idx_holdout])),np.arange(len(train)))
assert np.array_equal(train.raw_row.to_numpy()[idx_development],split['raw_train_idx'])
assert np.array_equal(train.raw_row.to_numpy()[idx_holdout],split['raw_valid_idx'])
assert not set(train.normalized.iloc[idx_development])&set(train.normalized.iloc[idx_holdout])
group_counts=np.zeros((len(unique_texts),6),dtype=np.int64)
np.add.at(group_counts,(group_ids,y_all),1)
group_targets=group_counts.argmax(axis=1)
groups_development=group_ids[idx_development]
group_folds=list(StratifiedKFold(5,shuffle=True,random_state=SEED).split(dev_groups,group_targets[dev_groups]))
folds=[]
leak_rows=[]
for number,(fit_positions,valid_positions) in enumerate(group_folds,1):
    fit_idx=idx_development[np.flatnonzero(np.isin(groups_development,dev_groups[fit_positions]))]
    valid_idx=idx_development[np.flatnonzero(np.isin(groups_development,dev_groups[valid_positions]))]
    overlap=set(train.normalized.iloc[fit_idx])&set(train.normalized.iloc[valid_idx])
    assert not overlap,('Fuga de texto normalizado',number)
    lower_overlap={text.lower() for text in train.normalized.iloc[fit_idx]}&{text.lower() for text in train.normalized.iloc[valid_idx]}
    leak_rows.append(dict(pliegue=number,filas_entrenamiento=len(fit_idx),filas_validacion=len(valid_idx),exactos=len(overlap),minusculas=len(lower_overlap)))
    folds.append((fit_idx,valid_idx))
print('VERIFICACION PREVIA COMPLETADA; todavia no se ha entrenado.')
print('Filas train/test/plantilla:',len(train_raw),len(test_raw),len(sample_raw))
print('Train original/limpio/excluidos:',len(train_raw),len(train),int(excluded.sum()))
print('Desarrollo/holdout:',len(idx_development),len(idx_holdout),'| Solo URL:',int(train.normalized.eq('<url>').sum()))
print('SHA-256 train:',TRAIN_SHA)
print('SHA-256 NPZ v6:',SPLIT_SHA)
print('Hash train coincidente con el NPZ de v6; no se regenero ninguna particion.')
display(pd.DataFrame(leak_rows))
RUN_FINGERPRINT=hashlib.sha256(json.dumps(dict(implementation=IMPLEMENTATION_VERSION,params=PARAMS,seed=SEED,train=TRAIN_SHA,split=SPLIT_SHA,classes=CLASSES.tolist()),sort_keys=True).encode()).hexdigest()

VERIFICACION PREVIA COMPLETADA; todavia no se ha entrenado.
Filas train/test/plantilla: 169336 112892 112892
Train original/limpio/excluidos: 169336 169181 155
Desarrollo/holdout: 133065 36116 | Solo URL: 3785
SHA-256 train: 39c6c8270366fe155f3c8dd232e31867bee659af8908121d04b6bd89bd163748
SHA-256 NPZ v6: fb299a3ef97a2a6948685163ba80aef5705c442e9ccda66c2c1fb0bce1e43c4d
Hash train coincidente con el NPZ de v6; no se regenero ninguna particion.


,pliegue,filas_entrenamiento,filas_validacion,exactos,minusculas
0,1,106546,26519,0,61
1,2,106411,26654,0,62
2,3,106086,26979,0,68
3,4,106438,26627,0,71
4,5,106779,26286,0,53


## Checkpoints seguros y reutilizables en Drive
Cada cache queda ligado a hashes, parametros e indices. Si cambia la procedencia, se detiene en vez de sobrescribirla. Se guarda el estado de cada epoca, el mejor modelo y el estado de Adam/RNG para reanudar; los checkpoints terminados se cargan sin reentrenar. Son pesos entrenados por este experimento, no modelos externos.

In [5]:
def fingerprint(stage,**items):
    return hashlib.sha256(json.dumps(dict(run=RUN_FINGERPRINT,stage=stage,**items),sort_keys=True).encode()).hexdigest()

def index_digest(indices):
    return hashlib.sha256(np.asarray(indices,dtype='<i8').tobytes()).hexdigest()

def atomic_torch_save(value,path):
    temporary=path.with_suffix(path.suffix+'.partial')
    torch.save(value,temporary)
    temporary.replace(path)

def save_npz_checked(path,**values):
    temporary=path.with_suffix(path.suffix+'.partial')
    with temporary.open('wb') as handle:np.savez_compressed(handle,**values)
    temporary.replace(path)

def load_npz_checked(path,signature):
    if not path.exists():return None
    with np.load(path,allow_pickle=False) as handle:payload={name:handle[name].copy() for name in handle.files}
    assert str(payload['fingerprint'])==signature,f'Cache incompatible: {path.name}; use otro RUN_ID.'
    return payload

def capture_rng():
    numpy_state=np.random.get_state()
    return dict(python=random.getstate(),numpy=[numpy_state[0],numpy_state[1].tolist(),numpy_state[2],numpy_state[3],numpy_state[4]],torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])

def restore_rng(value):
    random.setstate(value['python'])
    state=value['numpy']
    np.random.set_state((state[0],np.asarray(state[1],dtype=np.uint32),state[2],state[3],state[4]))
    torch.set_rng_state(value['torch'].cpu())
    if torch.cuda.is_available() and value['cuda']:torch.cuda.set_rng_state_all([item.cpu() for item in value['cuda']])


## Referencia v6 recalculada en los mismos pliegues
Se copia ModifiedKN y se fija la configuracion (4,3,0.75), sin sesgos. El valor historico 0.44106 no sustituye las predicciones pareadas. Las predicciones KN de cada pliegue se guardan tambien en Drive.

In [6]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

def predict_kn(fit_idx,app_idx,stage):
    signature=fingerprint('kn',stage_name=stage,fit=index_digest(fit_idx),app=index_digest(app_idx))
    path=RUN_DIR/f'{stage}_v6_predictions.npz'
    cached=load_npz_checked(path,signature)
    if cached is not None:
        assert np.array_equal(cached['indices'],app_idx)
        print('Referencia KN recuperada:',path.name)
        return cached['predictions']
    fit_texts=train.normalized.iloc[fit_idx].to_numpy()
    app_texts=train.normalized.iloc[app_idx].to_numpy()
    scores=np.zeros((len(app_idx),6))
    for level,order,weight in [('char',4,0.75),('word',3,0.25)]:
        mapping=make_encoder(fit_texts,level)
        fit_encoded=[encode_text(text,level,mapping) for text in fit_texts]
        app_encoded=[encode_text(text,level,mapping) for text in app_texts]
        for class_index in range(6):
            model=ModifiedKN(order,set(mapping.values())).fit(sequence for sequence,target in zip(fit_encoded,y_all[fit_idx]) if target==class_index)
            scores[:,class_index]+=weight*np.array([model.log_probability(sequence,order) for sequence in app_encoded])
            model.close();del model;gc.collect()
    scores+=np.log(np.bincount(y_all[fit_idx],minlength=6)/len(fit_idx))
    predictions=scores.argmax(axis=1)
    save_npz_checked(path,indices=app_idx,predictions=predictions,fingerprint=np.array(signature))
    return predictions


## Caracteres, lotes por longitud y padding
Vocabulario compartido entre clases, construido solo con el 90 % de ajuste interno del pliegue. Los IDs son 0 pad, 1 unk, 2 bos y 3 eos. Se truncan **140 caracteres de contenido**, luego se agregan BOS/EOS: como maximo 142 simbolos. El objetivo es el siguiente caracter, incluido EOS. Se ordenan por longitud los ejemplos de cada lote y se permuta solo el orden de los lotes de entrenamiento con semilla determinista.

In [7]:
def build_char_vocab(texts):
    characters=sorted(set(character for text in texts for character in text[:PARAMS['max_chars']]))
    vocabulary={'<pad>':PAD_ID,'<unk>':UNK_ID,'<bos>':BOS_ID,'<eos>':EOS_ID}
    vocabulary.update({character:index+4 for index,character in enumerate(characters)})
    return vocabulary

def encode_char_text(text,vocabulary):
    return [BOS_ID]+[vocabulary.get(character,UNK_ID) for character in text[:PARAMS['max_chars']]]+[EOS_ID]

def make_records(texts,vocabulary):
    return [(index,encode_char_text(text,vocabulary)) for index,text in enumerate(texts)]

def collate_records(records):
    records=sorted(records,key=lambda item:len(item[1]),reverse=True)
    positions=torch.tensor([item[0] for item in records],dtype=torch.long)
    lengths=torch.tensor([len(item[1])-1 for item in records],dtype=torch.long)
    inputs=torch.full((len(records),int(lengths.max())),PAD_ID,dtype=torch.long)
    targets=torch.full_like(inputs,PAD_ID)
    for row,(_,sequence) in enumerate(records):
        inputs[row,:len(sequence)-1]=torch.tensor(sequence[:-1])
        targets[row,:len(sequence)-1]=torch.tensor(sequence[1:])
    return positions,inputs,targets,lengths

def make_loader(records,training=False,epoch=0):
    ordered=sorted(range(len(records)),key=lambda index:len(records[index][1]),reverse=True)
    batches=[ordered[start:start+PARAMS['batch']] for start in range(0,len(ordered),PARAMS['batch'])]
    if training:random.Random(SEED+epoch).shuffle(batches)
    return DataLoader(records,batch_sampler=batches,collate_fn=collate_records,num_workers=0,pin_memory=True)

def prebatch_encoded(sequences):
    records=list(enumerate(sequences))
    ordered=sorted(range(len(records)),key=lambda index:len(records[index][1]),reverse=True)
    prepared=[]
    for start in range(0,len(ordered),PARAMS['batch']):
        selected=[records[index] for index in ordered[start:start+PARAMS['batch']]]
        positions,inputs,targets,lengths=collate_records(selected)
        tokens=int(lengths.sum())
        assert tokens==int(targets.ne(PAD_ID).sum())
        prepared.append(dict(positions=positions,inputs=inputs,targets=targets,lengths=lengths,tokens=tokens))
    return prepared

def move_prepared_to_device(batches):
    prepared=[]
    for batch in batches:
        result=dict(batch)
        for key in ['positions','inputs','targets']:
            tensor=batch[key]
            if DEVICE.type=='cuda':tensor=tensor.pin_memory()
            result[key]=tensor.to(DEVICE,non_blocking=DEVICE.type=='cuda')
        result['lengths']=batch['lengths'].cpu()
        prepared.append(result)
    return prepared

def records_from_prepared(batches):
    count=sum(len(batch['lengths']) for batch in batches)
    records=[None]*count
    for batch in batches:
        for row,(position,length) in enumerate(zip(batch['positions'].tolist(),batch['lengths'].tolist())):
            sequence=batch['inputs'][row,:length].tolist()+[int(batch['targets'][row,length-1])]
            records[position]=(position,sequence)
    assert all(record is not None for record in records)
    return records

## LSTM generativa con identidad fija
Cada clase tiene su propio LSTM y softmax. La capa Embedding es exclusivamente una tabla identidad congelada, equivalente a one-hot; se comprueba al crear y al cargar cada modelo. El padding se excluye de la recurrencia mediante packing y de la perdida mediante ignore_index=0. No hay clasificador entrenado sobre scores.

In [8]:
class CharLSTM(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.vocab_size=vocab_size
        self.fixed_one_hot=nn.Embedding(vocab_size,vocab_size,padding_idx=PAD_ID)
        with torch.no_grad():self.fixed_one_hot.weight.copy_(torch.eye(vocab_size))
        self.fixed_one_hot.weight.requires_grad_(False)
        self.lstm=nn.LSTM(vocab_size,PARAMS['hidden'],num_layers=PARAMS['layers'],batch_first=True)
        self.output_dropout=nn.Dropout(PARAMS['dropout'])
        self.output=nn.Linear(PARAMS['hidden'],vocab_size)

    def check_one_hot(self):
        assert not self.fixed_one_hot.weight.requires_grad
        expected=torch.eye(self.vocab_size,device=self.fixed_one_hot.weight.device)
        assert torch.equal(self.fixed_one_hot.weight,expected)
        assert self.lstm.input_size==self.vocab_size and self.lstm.num_layers==1

    def forward(self,inputs,lengths):
        one_hot=self.fixed_one_hot(inputs)
        packed=pack_padded_sequence(one_hot,lengths.cpu(),batch_first=True,enforce_sorted=True)
        encoded,_=self.lstm(packed)
        padded,_=pad_packed_sequence(encoded,batch_first=True,total_length=inputs.size(1))
        return self.output(self.output_dropout(padded))

def token_loss(logits,targets):
    return F.cross_entropy(logits.reshape(-1,logits.size(-1)),targets.reshape(-1),ignore_index=PAD_ID,reduction='sum')

def epoch_loss(model,records,optimizer=None,epoch=0):
    training=optimizer is not None
    model.train(training)
    total_loss,total_tokens=0.0,0
    with torch.set_grad_enabled(training):
        for _,inputs,targets,lengths in make_loader(records,training,epoch):
            inputs,targets=inputs.to(DEVICE),targets.to(DEVICE)
            if training:optimizer.zero_grad(set_to_none=True)
            logits=model(inputs,lengths)
            summed=token_loss(logits,targets)
            tokens=int(targets.ne(PAD_ID).sum())
            loss=summed/tokens
            if not torch.isfinite(loss):raise ArithmeticError('NLL no finita: detener, no saltar la celda.')
            if training:loss.backward();optimizer.step()
            total_loss+=float(summed.detach().cpu());total_tokens+=tokens
    assert total_tokens>0
    return total_loss/total_tokens

def score_lstm(model,texts,vocabulary):
    records=make_records(texts,vocabulary)
    scores=np.empty(len(texts),dtype=np.float64)
    model.eval()
    with torch.inference_mode():
        for positions,inputs,targets,lengths in make_loader(records):
            inputs,targets=inputs.to(DEVICE),targets.to(DEVICE)
            log_probs=F.log_softmax(model(inputs,lengths),dim=-1)
            selected=log_probs.gather(-1,targets.unsqueeze(-1)).squeeze(-1)
            selected=selected.masked_fill(targets.eq(PAD_ID),0.0)
            scores[positions.numpy()]=selected.sum(dim=1).cpu().numpy()
    assert np.isfinite(scores).all()
    return scores


## Early stopping interno y reanudacion
El 10 % interno se separa por grupos, estratificados por su clase mayoritaria, solo dentro del entrenamiento exterior. No se consulta el pliegue exterior. El vocabulario usa el 90 % restante; cada LSTM aprende solo su clase. Se conserva la mejor NLL de caracteres interna, paciencia 3 y maximo 30 epocas. Una cache terminada evita entrenar otra vez; una parcial recupera pesos, Adam y RNG al final de la ultima epoca.

In [9]:
def inner_split(fit_idx):
    fit_groups=np.unique(group_ids[fit_idx])
    core_groups,inner_groups=train_test_split(fit_groups,test_size=PARAMS['inner_fraction'],stratify=group_targets[fit_groups],random_state=SEED)
    core_idx=fit_idx[np.isin(group_ids[fit_idx],core_groups)]
    inner_idx=fit_idx[np.isin(group_ids[fit_idx],inner_groups)]
    assert not set(train.normalized.iloc[core_idx])&set(train.normalized.iloc[inner_idx])
    assert set(y_all[core_idx])==set(range(6)) and set(y_all[inner_idx])==set(range(6))
    return core_idx,inner_idx

def cpu_state(model):
    return {name:value.detach().cpu().clone() for name,value in model.state_dict().items()}

def train_class(core_idx,inner_idx,class_index,vocabulary,stage):
    core_class=core_idx[y_all[core_idx]==class_index]
    inner_class=inner_idx[y_all[inner_idx]==class_index]
    assert len(core_class)>0 and len(inner_class)>0
    signature=fingerprint('lstm',stage_name=stage,clase=class_index,core=index_digest(core_class),inner=index_digest(inner_class),vocabulary=vocabulary)
    path=RUN_DIR/f'{stage}_class_{CLASSES[class_index]}.pt'
    seed_everything()
    model=CharLSTM(len(vocabulary)).to(DEVICE)
    model.check_one_hot()
    optimizer=torch.optim.Adam([parameter for parameter in model.parameters() if parameter.requires_grad],lr=PARAMS['lr'])
    best_loss,bad_epochs,start_epoch,best_epoch=float('inf'),0,0,0
    best_state=None
    history=[]
    if path.exists():
        checkpoint=torch.load(path,map_location='cpu',weights_only=True)
        assert checkpoint['fingerprint']==signature,f'Checkpoint incompatible {path.name}; use otro RUN_ID.'
        assert checkpoint['vocabulary']==vocabulary
        history=checkpoint['history']
        best_loss,bad_epochs,best_epoch=checkpoint['best_loss'],checkpoint['bad_epochs'],checkpoint['best_epoch']
        best_state=checkpoint['best_state']
        if checkpoint['complete']:
            model.load_state_dict(best_state);model.check_one_hot()
            print('Checkpoint terminado recuperado:',path.name,'mejor epoca',best_epoch)
            return model,history
        model.load_state_dict(checkpoint['last_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        start_epoch=checkpoint['epoch']+1
        restore_rng(checkpoint['rng'])
        print('Reanudando:',path.name,'epoca',start_epoch+1)
    core_records=make_records(train.normalized.iloc[core_class].tolist(),vocabulary)
    inner_records=make_records(train.normalized.iloc[inner_class].tolist(),vocabulary)
    for epoch in range(start_epoch,PARAMS['max_epochs']):
        started=time.perf_counter()
        training_loss=epoch_loss(model,core_records,optimizer,epoch)
        validation_loss=epoch_loss(model,inner_records)
        improved=validation_loss<best_loss
        if improved:
            best_loss=validation_loss;bad_epochs=0;best_epoch=epoch+1;best_state=cpu_state(model)
        else:bad_epochs+=1
        history.append(dict(epoch=epoch+1,train_nll=training_loss,internal_nll=validation_loss,seconds=time.perf_counter()-started))
        complete=bad_epochs>=PARAMS['patience'] or epoch+1>=PARAMS['max_epochs']
        payload=dict(fingerprint=signature,vocabulary=vocabulary,epoch=epoch,best_epoch=best_epoch,best_loss=best_loss,bad_epochs=bad_epochs,complete=complete,last_state=cpu_state(model),best_state=best_state,optimizer_state=optimizer.state_dict(),rng=capture_rng(),history=history)
        atomic_torch_save(payload,path)
        print(stage,str(CLASSES[class_index]),'epoca',epoch+1,'NLL ajuste/interna',round(training_loss,4),round(validation_loss,4),'paciencia',bad_epochs,flush=True)
        if complete:break
    assert best_state is not None
    model.load_state_dict(best_state);model.check_one_hot()
    return model,history

def predict_neural(fit_idx,app_texts,stage):
    core_idx,inner_idx=inner_split(fit_idx)
    vocabulary=build_char_vocab(train.normalized.iloc[core_idx])
    assert [vocabulary[token] for token in ['<pad>','<unk>','<bos>','<eos>']]==[0,1,2,3]
    scores=np.zeros((len(app_texts),6),dtype=np.float64)
    for class_index in range(6):
        model,history=train_class(core_idx,inner_idx,class_index,vocabulary,stage)
        scores[:,class_index]=score_lstm(model,app_texts,vocabulary)
        display(pd.DataFrame(history).tail(4))
        del model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    scores+=np.log(np.bincount(y_all[fit_idx],minlength=6)/len(fit_idx))
    return scores.argmax(axis=1)


## Guardia de cumplimiento y pruebas antes de entrenar
La guardia permite exclusivamente la identidad fija declarada; prohíbe vectorizadores, cargas de pesos externos, transformers y embeddings aprendidos. Las comprobaciones numericas verifican la matriz identidad congelada, limites de longitud, padding, EOS, softmax y equivalencia de scores por lote o por ejemplo. Estos son casos sinteticos, no entrenamiento del corpus.

In [10]:
def ast_guard(source_cells):
    forbidden={'tfidfvectorizer','countvectorizer','hashingvectorizer','dictvectorizer','feature_extraction','linear_model','transformer','transformers','word2vec','glove','automodel','autotokenizer'}
    findings=[]
    for block_number,source in enumerate(source_cells,1):
        for node in ast.walk(ast.parse(source)):
            names=[]
            if isinstance(node,ast.Name):names=[node.id]
            elif isinstance(node,ast.Attribute):names=[node.attr]
            elif isinstance(node,ast.Import):names=[alias.name for alias in node.names]
            elif isinstance(node,ast.ImportFrom):names=[node.module or '']+[alias.name for alias in node.names]
            for name in names:
                if forbidden.intersection(name.lower().split('.')):findings.append((block_number,node.lineno,name))
            if isinstance(node,ast.Call):
                if isinstance(node.func,ast.Name) and node.func.id in {'eval','exec','__import__'}:findings.append((block_number,node.lineno,'ejecucion dinamica'))
                if isinstance(node.func,ast.Attribute) and node.func.attr=='from_pretrained':findings.append((block_number,node.lineno,'carga preentrenada'))
    if findings:raise RuntimeError(f'Guardia AST: usos prohibidos {findings}')
    return True

def compliance_checks():
    model=CharLSTM(8).to(DEVICE)
    model.check_one_hot()
    assert not model.fixed_one_hot.weight.requires_grad
    assert all(parameter.requires_grad for name,parameter in model.named_parameters() if name!='fixed_one_hot.weight')
    vocabulary={'<pad>':0,'<unk>':1,'<bos>':2,'<eos>':3,'a':4,'b':5,' ':6,'!':7}
    assert len(encode_char_text('a'*200,vocabulary))==142
    assert encode_char_text('z',vocabulary)==[2,1,3]
    records=make_records(['ab!','a'],vocabulary)
    positions,inputs,targets,lengths=collate_records(records)
    model.eval()
    with torch.no_grad():
        logits=model(inputs.to(DEVICE),lengths)
        assert torch.allclose(F.softmax(logits,dim=-1).sum(-1),torch.ones_like(logits[...,0]),atol=1e-6)
        loss=token_loss(logits,targets.to(DEVICE))
        mask=targets.to(DEVICE).ne(PAD_ID)
        direct=F.cross_entropy(logits[mask],targets.to(DEVICE)[mask],reduction='sum')
        assert torch.allclose(loss,direct,atol=1e-6)
    batched=score_lstm(model,['ab!','a'],vocabulary)
    separate=np.array([score_lstm(model,[text],vocabulary)[0] for text in ['ab!','a']])
    assert np.allclose(batched,separate,atol=1e-5)
    identity_before=model.fixed_one_hot.weight.detach().clone()
    optimizer=torch.optim.Adam([parameter for parameter in model.parameters() if parameter.requires_grad],lr=PARAMS['lr'])
    optimizer.zero_grad(set_to_none=True)
    model.train();token_loss(model(inputs.to(DEVICE),lengths),targets.to(DEVICE)).backward();optimizer.step()
    assert torch.equal(identity_before,model.fixed_one_hot.weight) and model.fixed_one_hot.weight.grad is None
    del model
    seed_everything()
    print('Pruebas one-hot, padding, EOS, truncado, softmax, orden de scores y congelacion: OK.')

from IPython import get_ipython
shell=get_ipython()
assert shell is not None,'Ejecute en Google Colab/IPython para verificar las fuentes antes de entrenar.'
executed_sources=[text for text in shell.history_manager.input_hist_raw if text.strip()]
assert ast_guard(executed_sources)
print('Guardia AST superada; Embedding permitida solo como identidad fija, verificada numericamente a continuacion.')
compliance_checks()

Guardia AST superada; Embedding permitida solo como identidad fija, verificada numericamente a continuacion.
Pruebas one-hot, padding, EOS, truncado, softmax, orden de scores y congelacion: OK.


## Trabajos independientes y tensores preparados

La ruta de ejecucion de CV/holdout/final usa procesos Python nuevos por par (etapa, clase). El pequeno ThreadPool del orquestador solo lanza y espera subprocesos; no entrena modelos ni comparte CUDA. Cada proceso inicializa `cuda:0`, mantiene su RNG privado con **semilla 42** (igual que el bucle secuencial original) y escribe checkpoint, scores, metricas y log exclusivos en Drive. Los fallos se recogen despues de que terminen los demas trabajos; una etapa con fallos no se marca completa.

Se codifican una vez los textos de cada etapa y se guarda un archivo de lotes CPU de solo lectura. Cada proceso copia sus lotes a GPU antes de las epocas. Las longitudes quedan en CPU porque `pack_padded_sequence` las requiere alli; asi `lengths.cpu()` no causa transferencias GPU por lote. No se hace collate ni se copian inputs/targets dentro del bucle de entrenamiento. Si se usa el cargador de referencia, tiene `pin_memory=True`.

La perdida se acumula en GPU y se transfieren conjuntamente las dos NLL una vez por epoca. Los checkpoints al final de epoca siguen requiriendo transferir y guardar pesos/Adam/RNG: no se afirma que desaparezca ese coste. Se conserva el avance RNG que hacia el iterador DataLoader para que dropout y reanudacion coincidan. Los bucles secuenciales anteriores permanecen exclusivamente como referencia de equivalencia, no como ruta principal.

Controles visibles: `N_PARALELO=6`, `DETERMINISTA=True`, `PRECISION='fp32'`. TF32 esta apagado tanto en matmul como en cuDNN; no se activa AMP, bf16 ni fp16. Cambiar determinismo exige una decision explicita y se registra; un checkpoint con otra politica numerica se rechaza. El worker se exporta como `v9_gpu_worker.py` dentro del directorio del experimento para funcionar con Colab sin depender de pickling de funciones del notebook.

In [11]:
def gpu_snapshot(label):
    print('NVIDIA-SMI:',label,flush=True)
    try:
        result=subprocess.run(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used,memory.total','--format=csv'],capture_output=True,text=True,check=True)
        print(result.stdout,flush=True)
        return result.stdout
    except (FileNotFoundError,subprocess.CalledProcessError) as error:
        print('nvidia-smi no disponible:',str(error),flush=True)
        return 'no disponible'

def atomic_json_save(value,path):
    temporary=path.with_suffix(path.suffix+'.partial')
    with temporary.open('w',encoding='utf-8') as handle:json.dump(value,handle,ensure_ascii=False,indent=2)
    temporary.replace(path)

def prepared_epoch_loss(model,batches,optimizer=None,epoch=0):
    torch.empty((),dtype=torch.int64).random_()
    training=optimizer is not None
    model.train(training)
    order=list(range(len(batches)))
    if training:random.Random(SEED+epoch).shuffle(order)
    total=torch.zeros((),device=DEVICE,dtype=torch.float64)
    tokens=sum(batch['tokens'] for batch in batches)
    if tokens<=0:raise ValueError('No hay tokens validos en los lotes.')
    with torch.set_grad_enabled(training):
        for position in order:
            batch=batches[position]
            if training:optimizer.zero_grad(set_to_none=True)
            logits=model(batch['inputs'],batch['lengths'])
            summed=token_loss(logits,batch['targets'])
            if training:
                (summed/batch['tokens']).backward()
                optimizer.step()
            total.add_(summed.detach().to(torch.float64))
    return total/tokens

def prepared_scores(model,batches,count,return_tokens=False):
    torch.empty((),dtype=torch.int64).random_()
    scores=torch.empty(count,device=DEVICE,dtype=torch.float32)
    token_predictions=torch.full((count,max(batch['targets'].size(1) for batch in batches)),-1,device=DEVICE,dtype=torch.long) if return_tokens else None
    model.eval()
    with torch.inference_mode():
        for batch in batches:
            logits=model(batch['inputs'],batch['lengths'])
            log_probs=F.log_softmax(logits,dim=-1)
            selected=log_probs.gather(-1,batch['targets'].unsqueeze(-1)).squeeze(-1)
            selected=selected.masked_fill(batch['targets'].eq(PAD_ID),0.0)
            scores[batch['positions']]=selected.sum(dim=1)
            if return_tokens:
                values=logits.argmax(dim=-1).masked_fill(batch['targets'].eq(PAD_ID),-1)
                token_predictions[batch['positions'],:values.size(1)]=values
    return scores.cpu().numpy().astype(np.float64),token_predictions.cpu().numpy() if return_tokens else np.empty((0,0),dtype=np.int64)

def legacy_prepared_scores(model,cpu_batches,count,return_tokens=False):
    records=records_from_prepared(cpu_batches)
    scores=np.empty(count,dtype=np.float64)
    width=max(batch['targets'].size(1) for batch in cpu_batches)
    token_predictions=np.full((count,width),-1,dtype=np.int64) if return_tokens else np.empty((0,0),dtype=np.int64)
    model.eval()
    with torch.inference_mode():
        for positions,inputs,targets,lengths in make_loader(records):
            inputs,targets=inputs.to(DEVICE),targets.to(DEVICE)
            logits=model(inputs,lengths)
            values=F.log_softmax(logits,dim=-1).gather(-1,targets.unsqueeze(-1)).squeeze(-1)
            values=values.masked_fill(targets.eq(PAD_ID),0.0)
            scores[positions.numpy()]=values.sum(dim=1).cpu().numpy()
            if return_tokens:
                predicted=logits.argmax(dim=-1).masked_fill(targets.eq(PAD_ID),-1).cpu().numpy()
                token_predictions[positions.numpy(),:predicted.shape[1]]=predicted
    return scores,token_predictions

def execute_worker(job):
    global DEVICE,DETERMINISTA
    started=time.perf_counter()
    checkpoint_path=Path(job['checkpoint'])
    result_path=Path(job['result'])
    metrics_path=Path(job['metrics'])
    policy=dict(determinista=bool(job['determinista']),precision='fp32',tf32=False,seed=int(job['seed']))
    checkpoint=torch.load(checkpoint_path,map_location='cpu',weights_only=True) if checkpoint_path.exists() else None
    if checkpoint is not None:
        assert checkpoint['fingerprint']==job['checkpoint_signature'],'Checkpoint de otro ajuste; use otro RUN_ID.'
        old_policy=checkpoint.get('numeric_policy',dict(determinista=True,precision='fp32',tf32=False,seed=42))
        assert old_policy==policy,'Politica numerica diferente; use otro RUN_ID, no se reentrena silenciosamente.'
    if checkpoint is not None and checkpoint['complete'] and result_path.exists() and metrics_path.exists():
        with metrics_path.open(encoding='utf-8') as handle:previous=json.load(handle)
        with np.load(result_path,allow_pickle=False) as values:
            valid=str(values['fingerprint'])==job['result_signature']
        if previous.get('ok') and previous.get('fingerprint')==job['result_signature'] and valid:
            print('TRABAJO COMPLETO RECUPERADO SIN REENTRENAR:',job['id'],flush=True)
            return previous
    DETERMINISTA=policy['determinista']
    torch.set_num_threads(int(job['cpu_threads']))
    DEVICE=torch.device('cuda:0' if job['device']=='cuda' else job['device'])
    if DEVICE.type=='cuda':
        if not torch.cuda.is_available():raise RuntimeError('El trabajo requiere CUDA; no se cambia a CPU silenciosamente.')
        torch.cuda.set_device(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)
    seed_everything(policy['seed'])
    data=torch.load(job['inputs'],map_location='cpu',weights_only=True)
    assert data['fingerprint']==job['inputs_signature']
    vocabulary=data['vocabulary']
    class_data=data['classes'][int(job['class_index'])]
    cpu_application=data['application']
    count=data['application_count']
    if job['mode']=='preloaded':
        core=move_prepared_to_device(class_data['core'])
        internal=move_prepared_to_device(class_data['internal'])
        application=move_prepared_to_device(cpu_application)
        core_records,internal_records=None,None
    else:
        core_records=records_from_prepared(class_data['core'])
        internal_records=records_from_prepared(class_data['internal'])
        core,internal,application=None,None,None
    del data
    seed_everything(policy['seed'])
    model=CharLSTM(len(vocabulary)).to(DEVICE)
    model.check_one_hot()
    optimizer=torch.optim.Adam([parameter for parameter in model.parameters() if parameter.requires_grad],lr=PARAMS['lr'])
    best_loss,bad_epochs,start_epoch,best_epoch=float('inf'),0,0,0
    best_state=None
    history=[]
    reused=False
    if checkpoint is not None:
        assert checkpoint['vocabulary']==vocabulary
        history=checkpoint['history']
        best_loss,bad_epochs,best_epoch=checkpoint['best_loss'],checkpoint['bad_epochs'],checkpoint['best_epoch']
        best_state=checkpoint['best_state']
        if checkpoint['complete']:
            reused=True
        else:
            model.load_state_dict(checkpoint['last_state'])
            optimizer.load_state_dict(checkpoint['optimizer_state'])
            start_epoch=checkpoint['epoch']+1
            restore_rng(checkpoint['rng'])
    limit=2 if job.get('fixture') else PARAMS['max_epochs']
    if not reused:
        for epoch in range(start_epoch,limit):
            epoch_started=time.perf_counter()
            if job['mode']=='legacy':
                training_loss=epoch_loss(model,core_records,optimizer,epoch)
                validation_loss=epoch_loss(model,internal_records)
            else:
                train_value=prepared_epoch_loss(model,core,optimizer,epoch)
                internal_value=prepared_epoch_loss(model,internal)
                training_loss,validation_loss=torch.stack((train_value,internal_value)).detach().cpu().tolist()
            if not math.isfinite(training_loss) or not math.isfinite(validation_loss):raise ArithmeticError('NLL no finita al terminar la epoca; el trabajo se marca fallido.')
            if validation_loss<best_loss:
                best_loss=validation_loss;bad_epochs=0;best_epoch=epoch+1;best_state=cpu_state(model)
            else:bad_epochs+=1
            history.append(dict(epoch=epoch+1,train_nll=training_loss,internal_nll=validation_loss,seconds=time.perf_counter()-epoch_started))
            complete=bad_epochs>=PARAMS['patience'] or epoch+1>=limit
            payload=dict(fingerprint=job['checkpoint_signature'],vocabulary=vocabulary,epoch=epoch,best_epoch=best_epoch,best_loss=best_loss,bad_epochs=bad_epochs,complete=complete,last_state=cpu_state(model),best_state=best_state,optimizer_state=optimizer.state_dict(),rng=capture_rng(),history=history,numeric_policy=policy)
            atomic_torch_save(payload,checkpoint_path)
            print(job['id'],'epoca',epoch+1,'NLL',training_loss,validation_loss,'DETERMINISTA',DETERMINISTA,flush=True)
            if epoch+1==max(1,limit//2):gpu_snapshot(job['id']+' mitad de epocas previstas')
            if complete:break
    assert best_state is not None
    model.load_state_dict(best_state);model.check_one_hot()
    if job['mode']=='legacy':scores,tokens=legacy_prepared_scores(model,cpu_application,count,job.get('return_tokens',False))
    else:scores,tokens=prepared_scores(model,application,count,job.get('return_tokens',False))
    with result_path.with_suffix('.npz.partial').open('wb') as handle:
        np.savez_compressed(handle,fingerprint=np.array(job['result_signature']),scores=scores,token_predictions=tokens)
    result_path.with_suffix('.npz.partial').replace(result_path)
    peak=int(torch.cuda.max_memory_allocated(DEVICE)) if DEVICE.type=='cuda' else 0
    metrics=dict(ok=True,id=job['id'],fingerprint=job['result_signature'],class_index=job['class_index'],seconds=time.perf_counter()-started,epochs=len(history),best_epoch=best_epoch,peak_memory_bytes=peak,history=history,reused_checkpoint=reused,numeric_policy=policy,device=str(DEVICE),gpu_name=torch.cuda.get_device_name(DEVICE) if DEVICE.type=='cuda' else 'cpu',pid=os.getpid())
    atomic_json_save(metrics,metrics_path)
    return metrics

def worker_entry(job_path):
    with Path(job_path).open(encoding='utf-8') as handle:job=json.load(handle)
    try:
        execute_worker(job)
    except Exception as error:
        import traceback
        details=dict(ok=False,id=job['id'],error=str(error),traceback=traceback.format_exc())
        atomic_json_save(details,Path(job['metrics']))
        raise

def export_worker(source_cells,destination):
    requested={'seed_everything','CharLSTM','token_loss','collate_records','make_loader','records_from_prepared','move_prepared_to_device','epoch_loss','capture_rng','restore_rng','cpu_state','atomic_torch_save','gpu_snapshot','atomic_json_save','prepared_epoch_loss','prepared_scores','legacy_prepared_scores','execute_worker','worker_entry'}
    definitions={}
    for source in source_cells:
        for node in ast.parse(source).body:
            if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in requested:
                definitions[node.name]=ast.get_source_segment(source,node)
    assert set(definitions)==requested,sorted(requested-set(definitions))
    imports="import os\nos.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'\nimport sys,json,math,time,random,subprocess\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.nn.utils.rnn import pack_padded_sequence,pad_packed_sequence\nfrom torch.utils.data import DataLoader\n"
    constants=dict(SEED=SEED,PARAMS=PARAMS,PAD_ID=PAD_ID,UNK_ID=UNK_ID,BOS_ID=BOS_ID,EOS_ID=EOS_ID,CLASSES=CLASSES.tolist(),DETERMINISTA=DETERMINISTA)
    content=imports+'\n'.join(f'{key}={value!r}' for key,value in constants.items())+'\n\n'+'\n\n'.join(definitions.values())+'\n\nif __name__=="__main__":\n    worker_entry(sys.argv[1])\n'
    compile(content,'worker-module','exec')
    assert ast_guard([content])
    destination.parent.mkdir(parents=True,exist_ok=True)
    destination.write_text(content,encoding='utf-8')
    return hashlib.sha256(content.encode()).hexdigest()

def prepare_stage_inputs(fit_idx,app_texts,stage):
    core_idx,inner_idx=inner_split(fit_idx)
    vocabulary=build_char_vocab(train.normalized.iloc[core_idx])
    signature=fingerprint('prebatched_stage_v1',stage_name=stage,core=index_digest(core_idx),inner=index_digest(inner_idx),vocabulary=vocabulary,application=hashlib.sha256(json.dumps(app_texts,ensure_ascii=False,separators=(',',':')).encode()).hexdigest())
    input_path=RUN_DIR/f'{stage}_encoded_batches.pt'
    checkpoint_signatures={class_index:fingerprint('lstm',stage_name=stage,clase=class_index,core=index_digest(core_idx[y_all[core_idx]==class_index]),inner=index_digest(inner_idx[y_all[inner_idx]==class_index]),vocabulary=vocabulary) for class_index in range(6)}
    if input_path.exists():
        cached=torch.load(input_path,map_location='cpu',weights_only=True)
        assert cached['fingerprint']==signature,'Datos precomputados incompatibles; use otro RUN_ID.'
        del cached
        return dict(path=input_path,fingerprint=signature,checkpoint_signatures=checkpoint_signatures,count=len(app_texts))
    core_sequences=[encode_char_text(text,vocabulary) for text in train.normalized.iloc[core_idx]]
    inner_sequences=[encode_char_text(text,vocabulary) for text in train.normalized.iloc[inner_idx]]
    application=[encode_char_text(text,vocabulary) for text in app_texts]
    classes={}
    for class_index in range(6):
        classes[class_index]=dict(core=prebatch_encoded([sequence for sequence,target in zip(core_sequences,y_all[core_idx]) if target==class_index]),internal=prebatch_encoded([sequence for sequence,target in zip(inner_sequences,y_all[inner_idx]) if target==class_index]))
    payload=dict(fingerprint=signature,vocabulary=vocabulary,classes=classes,application=prebatch_encoded(application),application_count=len(application))
    atomic_torch_save(payload,input_path)
    print('Codificacion y padding realizados una vez:',input_path.name)
    return dict(path=input_path,fingerprint=signature,checkpoint_signatures=checkpoint_signatures,count=len(app_texts))

def make_job(descriptor,class_index,stage,output_dir,mode='preloaded',fixture=False,device='cuda'):
    output_dir.mkdir(parents=True,exist_ok=True)
    label=str(CLASSES[class_index])
    stem=f'{stage}_class_{label}'
    signature=fingerprint('class_scores_preloaded_v1',inputs=descriptor['fingerprint'],clase=class_index,seed=SEED,determinista=DETERMINISTA,precision=PRECISION,mode=mode,fixture=fixture,worker=WORKER_SHA)
    return dict(id=stem,class_index=class_index,inputs=str(descriptor['path']),inputs_signature=descriptor['fingerprint'],checkpoint_signature=descriptor['checkpoint_signatures'][class_index],checkpoint=str(output_dir/(stem+'.pt')),result=str(output_dir/(stem+'_scores.npz')),metrics=str(output_dir/(stem+'_metrics.json')),manifest=str(output_dir/(stem+'_job.json')),log=str(output_dir/(stem+'.log')),result_signature=signature,seed=SEED,determinista=DETERMINISTA,cpu_threads=CPU_THREADS_PER_JOB,device=device,mode=mode,fixture=fixture,return_tokens=fixture)

def launch_one_job(job):
    started=time.perf_counter()
    fresh_training=not Path(job['checkpoint']).exists()
    atomic_json_save(job,Path(job['manifest']))
    with Path(job['log']).open('a',encoding='utf-8') as handle:
        completed=subprocess.run([sys.executable,str(WORKER_PATH),job['manifest']],stdout=handle,stderr=subprocess.STDOUT)
    details={}
    if Path(job['metrics']).exists():
        with Path(job['metrics']).open(encoding='utf-8') as handle:details=json.load(handle)
    if completed.returncode!=0 or not details.get('ok'):
        return dict(ok=False,id=job['id'],error=details.get('error',f'Proceso termino con codigo {completed.returncode}'),log=job['log'])
    if details.get('fingerprint')!=job['result_signature']:
        return dict(ok=False,id=job['id'],error='Firma del resultado no coincide',log=job['log'])
    details['fresh_training']=fresh_training
    details['process_wall_s']=time.perf_counter()-started
    return details

def run_jobs(jobs,parallelism,report_dir,label):
    assert len({job['id'] for job in jobs})==len(jobs),'Trabajos duplicados; no se comparten checkpoints.'
    started=time.perf_counter()
    reports=[]
    snapshot_start=gpu_snapshot(label+' inicio')
    snapshot_mid=None
    with ThreadPoolExecutor(max_workers=parallelism) as executor:
        futures={executor.submit(launch_one_job,job):job for job in jobs}
        for future in as_completed(futures):
            job=futures[future]
            try:details=future.result()
            except Exception as error:details=dict(ok=False,id=job['id'],error=str(error),log=job['log'])
            reports.append(details)
            print('Trabajo terminado:',details['id'],'OK' if details['ok'] else 'FALLO',flush=True)
            if snapshot_mid is None and len(reports)>=max(1,len(jobs)//2):snapshot_mid=gpu_snapshot(label+' mitad de trabajos')
    report=dict(complete=all(item['ok'] for item in reports),label=label,parallelism=parallelism,wall_seconds=time.perf_counter()-started,jobs=reports,nvidia_start=snapshot_start,nvidia_mid=snapshot_mid)
    report_dir.mkdir(parents=True,exist_ok=True)
    atomic_json_save(report,report_dir/'orchestrator_report.json')
    failed=[item for item in reports if not item['ok']]
    display(pd.DataFrame(reports).drop(columns=['history'],errors='ignore'))
    if failed:raise RuntimeError('Trabajos fallidos; los demas continuaron y no se marca completo: '+json.dumps(failed,ensure_ascii=False))
    return report

def load_job_output(job):
    with np.load(job['result'],allow_pickle=False) as handle:
        assert str(handle['fingerprint'])==job['result_signature']
        return {name:handle[name].copy() for name in handle.files}

def predict_neural(fit_idx,app_texts,stage):
    assert PARALLEL_READY,'Ejecute y supere las pruebas de equivalencia y el benchmark antes de la CV.'
    descriptor=prepare_stage_inputs(fit_idx,app_texts,stage)
    jobs=[make_job(descriptor,class_index,stage,RUN_DIR) for class_index in range(6)]
    report=run_jobs(jobs,N_EFECTIVO,RUN_DIR/f'{stage}_orchestrator',stage)
    EXECUTION_REPORTS.extend(report['jobs'])
    scores=np.column_stack([load_job_output(job)['scores'] for job in jobs])
    assert scores.shape==(len(app_texts),6) and np.isfinite(scores).all()
    scores+=np.log(np.bincount(y_all[fit_idx],minlength=6)/len(fit_idx))
    return scores.argmax(axis=1)

EXECUTION_REPORTS=[]
PARALLEL_READY=False
WORKER_PATH=RUN_DIR/'v9_gpu_worker.py'
source_history=[source for source in get_ipython().history_manager.input_hist_raw if source.strip()]
WORKER_SHA=export_worker(source_history,WORKER_PATH)
print('Worker exportado en Drive:',WORKER_PATH.name,'SHA-256',WORKER_SHA)
print('Procesos Python independientes; CUDA se inicializa dentro de cada proceso. Semilla privada 42, fp32, TF32 apagado y DETERMINISTA=',DETERMINISTA)

Worker exportado en Drive: v9_gpu_worker.py SHA-256 f5f2e5c82f4bbb319a44c9736036d07713c422cae4b94dd2a6703ba0248d6cec
Procesos Python independientes; CUDA se inicializa dentro de cada proceso. Semilla privada 42, fp32, TF32 apagado y DETERMINISTA= True


## Equivalencia obligatoria y benchmark del primer pliegue

Primero se compara una clase durante dos epocas: el bucle original secuencial contra dos procesos con lotes precargados. Se exigen diferencias absolutas de NLL y scores de como maximo **1e-5**, y predicciones de caracter identicas. Una discrepancia bloquea la CV; no se apaga determinismo ni se amplian tolerancias para eludirla.

Despues se ejecutan los seis trabajos del **pliegue 1 completo**, primero N=1 y luego N=6 por defecto. Se mantienen las 30 epocas maximas, paciencia 3 y todos los hiperparametros; no es una medida extrapolada del subconjunto. Se compara tambien el resultado de cada clase y la clasificacion conjunta entre las dos ejecuciones. Las carpetas de benchmark son distintas para no confundir una lectura de cache con entrenamiento fresco. Los pesos ganadores se reutilizan en CV para no entrenar por tercera vez el pliegue; checkpoints canonicos anteriores no se reemplazan.

Se imprimen tiempos por trabajo, epocas, tiempo total y `torch.cuda.max_memory_allocated` por proceso. `nvidia-smi` registra uso y memoria global al inicio y a mitad de los trabajos; los workers tambien lo registran a mitad de las epocas previstas. No se interpreta la suma de picos individuales como un pico global simultaneo.

Si el paralelo no mejora, **N_EFECTIVO=1** para el resto de CV/holdout/final y no se prueba mayor concurrencia. Si el benchmark se reanudo y no es fresco, se informa y tampoco habilita concurrencia mayor que uno. Un informe de benchmark ya completo puede recuperarse, conservando sus tiempos originales claramente registrados. No hay medicion A100 valida en CPU ni promesa de saturacion: el hardware y la aceleracion real se muestran en la salida.

In [12]:
def compare_job_results(reference_job,candidate_job,require_tokens=False):
    first=load_job_output(reference_job)
    second=load_job_output(candidate_job)
    with Path(reference_job['metrics']).open(encoding='utf-8') as handle:first_metrics=json.load(handle)
    with Path(candidate_job['metrics']).open(encoding='utf-8') as handle:second_metrics=json.load(handle)
    first_losses=np.array([[row['train_nll'],row['internal_nll']] for row in first_metrics['history']])
    second_losses=np.array([[row['train_nll'],row['internal_nll']] for row in second_metrics['history']])
    assert first_losses.shape==second_losses.shape,'Cambian las epocas/early stopping entre ejecuciones.'
    loss_error=float(np.max(np.abs(first_losses-second_losses)))
    score_error=float(np.max(np.abs(first['scores']-second['scores'])))
    assert loss_error<=1e-5,('Perdidas no equivalentes',loss_error)
    assert score_error<=1e-5,('Log P no equivalentes',score_error)
    if require_tokens:assert np.array_equal(first['token_predictions'],second['token_predictions']),'Predicciones de caracter distintas.'
    return dict(referencia=reference_job['id'],candidato=candidate_job['id'],error_max_perdida=loss_error,error_max_score=score_error,predicciones_caracter_iguales=bool(np.array_equal(first['token_predictions'],second['token_predictions'])) if require_tokens else None)

def first_encoded_sequences(batches,limit):
    result=[]
    for batch in batches:
        for row,length in enumerate(batch['lengths'].tolist()):
            result.append(batch['inputs'][row,:length].tolist()+[int(batch['targets'][row,length-1])])
            if len(result)>=limit:return result
    return result

def prepare_equivalence_inputs(descriptor,directory):
    directory.mkdir(parents=True,exist_ok=True)
    data=torch.load(descriptor['path'],map_location='cpu',weights_only=True)
    core=first_encoded_sequences(data['classes'][0]['core'],384)
    internal=first_encoded_sequences(data['classes'][0]['internal'],128)
    application=first_encoded_sequences(data['application'],64)
    signature=fingerprint('equivalence_fixture_v1',source=descriptor['fingerprint'],core=core,internal=internal,application=application)
    path=directory/'equivalence_inputs.pt'
    if path.exists():
        existing=torch.load(path,map_location='cpu',weights_only=True)
        assert existing['fingerprint']==signature,'Fixture incompatible; use otro BENCHMARK_ID.'
    else:
        atomic_torch_save(dict(fingerprint=signature,vocabulary=data['vocabulary'],classes={0:dict(core=prebatch_encoded(core),internal=prebatch_encoded(internal))},application=prebatch_encoded(application),application_count=len(application)),path)
    return dict(path=path,fingerprint=signature,checkpoint_signatures={0:fingerprint('fixture_checkpoint',inputs=signature)},count=len(application))

def run_equivalence_gate(descriptor,directory,device='cuda'):
    fixture=prepare_equivalence_inputs(descriptor,directory)
    legacy=make_job(fixture,0,'equivalence_legacy',directory/'legacy',mode='legacy',fixture=True,device=device)
    first=make_job(fixture,0,'equivalence_preloaded_1',directory/'parallel',fixture=True,device=device)
    second=make_job(fixture,0,'equivalence_preloaded_2',directory/'parallel',fixture=True,device=device)
    run_jobs([legacy],1,directory/'legacy_report','Equivalencia secuencial legacy, dos epocas')
    run_jobs([first,second],2,directory/'parallel_report','Equivalencia dos procesos privados, dos epocas')
    comparisons=[compare_job_results(legacy,first,True),compare_job_results(legacy,second,True),compare_job_results(first,second,True)]
    print('EQUIVALENCIA SUPERADA: perdidas y log P dentro de 1e-5; predicciones de caracter identicas. No se desactiva determinismo para forzar el resultado.')
    display(pd.DataFrame(comparisons))
    atomic_json_save(dict(complete=True,comparisons=comparisons,device=device),directory/'equivalence_report.json')
    return comparisons

def promote_benchmark_jobs(source_jobs,destination_jobs):
    import shutil
    for source_job,destination in zip(source_jobs,destination_jobs):
        if Path(destination['checkpoint']).exists():
            print('Se conserva checkpoint anterior; no se sustituye:',Path(destination['checkpoint']).name)
            continue
        for field in ['checkpoint','result','metrics']:
            source_path,destination_path=Path(source_job[field]),Path(destination[field])
            if destination_path.exists():continue
            temporary=destination_path.with_suffix(destination_path.suffix+'.promoting')
            shutil.copyfile(source_path,temporary)
            temporary.replace(destination_path)

def run_execution_preflight():
    global N_EFECTIVO,PARALLEL_READY,BENCHMARK_REPORT
    if DEVICE.type!='cuda':raise RuntimeError('La equivalencia y el benchmark de ejecucion completa requieren GPU de Colab. No se simula una A100 ni se cambia a CPU silenciosamente.')
    assert PRECISION=='fp32'
    torch.cuda.empty_cache()
    print('GPU real:',torch.cuda.get_device_name(0),'DETERMINISTA:',DETERMINISTA,'TF32: False')
    if 'A100' not in torch.cuda.get_device_name(0):print('Aviso: esta GPU no es A100; las medidas se etiquetan con el hardware real.')
    fit_idx,valid_idx=folds[0]
    descriptor=prepare_stage_inputs(fit_idx,train.normalized.iloc[valid_idx].tolist(),'fold_1')
    root=RUN_DIR/'execution_checks'/BENCHMARK_ID
    root.mkdir(parents=True,exist_ok=True)
    report_path=root/'benchmark_complete.json'
    signature=fingerprint('execution_benchmark_v1',inputs=descriptor['fingerprint'],worker=WORKER_SHA,gpu=torch.cuda.get_device_name(0),determinista=DETERMINISTA,precision=PRECISION,parallelism=min(6,N_PARALELO))
    if report_path.exists():
        with report_path.open(encoding='utf-8') as handle:report=json.load(handle)
        assert report['fingerprint']==signature and report['complete'],'Benchmark incompatible/incompleto; cambie BENCHMARK_ID, no los hiperparametros.'
        N_EFECTIVO=int(report['effective_parallelism'])
        BENCHMARK_REPORT=report
        PARALLEL_READY=True
        print('Benchmark real recuperado de Drive, sin repetir entrenamiento:',report_path.name)
        print('Aceleracion registrada:',report['speedup'],'concurrencia efectiva:',N_EFECTIVO)
        display(pd.DataFrame(report['table']))
        return report
    comparisons=run_equivalence_gate(descriptor,root/'equivalence','cuda')
    sequential_jobs=[make_job(descriptor,class_index,'fold_1',root/'sequential') for class_index in range(6)]
    parallel_jobs=[make_job(descriptor,class_index,'fold_1',root/'parallel') for class_index in range(6)]
    print('BENCHMARK COMPLETO DEL PLIEGUE 1: primero N=1, despues N=',min(6,N_PARALELO),'; maximo 30 epocas y early stopping sin cambios.')
    sequential=run_jobs(sequential_jobs,1,root/'sequential_report','Pliegue 1, N=1')
    parallel=run_jobs(parallel_jobs,min(6,N_PARALELO),root/'parallel_report','Pliegue 1, hasta seis procesos')
    full_comparisons=[compare_job_results(first,second) for first,second in zip(sequential_jobs,parallel_jobs)]
    sequential_scores=np.column_stack([load_job_output(job)['scores'] for job in sequential_jobs])
    parallel_scores=np.column_stack([load_job_output(job)['scores'] for job in parallel_jobs])
    prior=np.log(np.bincount(y_all[fit_idx],minlength=6)/len(fit_idx))
    assert np.array_equal((sequential_scores+prior).argmax(axis=1),(parallel_scores+prior).argmax(axis=1)),'La clasificacion completa del pliegue no es equivalente.'
    speedup=sequential['wall_seconds']/parallel['wall_seconds']
    comparable=all(item.get('fresh_training',False) for item in sequential['jobs']+parallel['jobs'])
    N_EFECTIVO=min(6,N_PARALELO) if comparable and speedup>1 and N_PARALELO>1 else 1
    if not comparable:print('Benchmark reanudado o con caches: las paredes actuales no son una medida fresca comparable. Se informa, y no se habilita paralelismo mayor que 1.')
    elif N_EFECTIVO==1:print('NO MEJORA EL PARALELISMO: aceleracion',speedup,'; el resto de CV, holdout y final usara N=1. No se prueba mas concurrencia.')
    else:print('Aceleracion real medida:',speedup,'| N efectivo:',N_EFECTIVO)
    table=[]
    for mode,measurement in [('N=1',sequential),('N='+str(min(6,N_PARALELO)),parallel)]:
        for row in measurement['jobs']:
            table.append(dict(modo=mode,trabajo=row['id'],epocas=row['epochs'],mejor_epoca=row['best_epoch'],tiempo_proceso_s=row['process_wall_s'],tiempo_trabajo_s=row['seconds'],memoria_max_MiB=row['peak_memory_bytes']/1048576,tiempo_total_s=measurement['wall_seconds'],fresco=row.get('fresh_training',False)))
    display(pd.DataFrame(table))
    print('Memoria max: torch.cuda.max_memory_allocated por proceso; no se interpreta la suma de picos como un pico global simultaneo. nvidia-smi muestra el estado global al inicio y a mitad.')
    winner=parallel_jobs if N_EFECTIVO>1 else sequential_jobs
    canonical=[make_job(descriptor,class_index,'fold_1',RUN_DIR) for class_index in range(6)]
    promote_benchmark_jobs(winner,canonical)
    report=dict(complete=True,fingerprint=signature,gpu=torch.cuda.get_device_name(0),determinista=DETERMINISTA,precision=PRECISION,tf32=False,sequential_seconds=sequential['wall_seconds'],parallel_seconds=parallel['wall_seconds'],speedup=speedup,fresh_measurement=comparable,effective_parallelism=N_EFECTIVO,table=table,small_equivalence=comparisons,fold_equivalence=full_comparisons)
    atomic_json_save(report,report_path)
    BENCHMARK_REPORT=report
    PARALLEL_READY=True
    return report

BENCHMARK_REPORT=run_execution_preflight()
print('PUERTA DE EJECUCION SUPERADA; la CV puede comenzar con N efectivo',N_EFECTIVO)

GPU real: NVIDIA A100-SXM4-40GB DETERMINISTA: True TF32: False
Codificacion y padding realizados una vez: fold_1_encoded_batches.pt
NVIDIA-SMI: Equivalencia secuencial legacy, dos epocas inicio
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 0 %, 640 MiB, 40960 MiB

Trabajo terminado: equivalence_legacy_class_13-17 OK
NVIDIA-SMI: Equivalencia secuencial legacy, dos epocas mitad de trabajos
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 0 %, 640 MiB, 40960 MiB



,ok,id,fingerprint,class_index,seconds,epochs,best_epoch,peak_memory_bytes,reused_checkpoint,numeric_policy,device,gpu_name,pid,fresh_training,process_wall_s
0,True,equivalence_legacy_class_13-17,389dcf3b92294633252fcfc0778d1e874cb4477f41884e...,0,2.204014,2,2,310465024,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,7230,True,8.319354


NVIDIA-SMI: Equivalencia dos procesos privados, dos epocas inicio
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 0 %, 640 MiB, 40960 MiB

Trabajo terminado: equivalence_preloaded_2_class_13-17 OK
NVIDIA-SMI: Equivalencia dos procesos privados, dos epocas mitad de trabajos
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 0 %, 640 MiB, 40960 MiB

Trabajo terminado: equivalence_preloaded_1_class_13-17 OK


,ok,id,fingerprint,class_index,seconds,epochs,best_epoch,peak_memory_bytes,reused_checkpoint,numeric_policy,device,gpu_name,pid,fresh_training,process_wall_s
0,True,equivalence_preloaded_2_class_13-17,ea80904ad893b5c42aa61c53106e6757e02e7ee7c265cc...,0,2.503633,2,2,311465472,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,7295,True,5.266078
1,True,equivalence_preloaded_1_class_13-17,ea80904ad893b5c42aa61c53106e6757e02e7ee7c265cc...,0,2.528904,2,2,311465472,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,7294,True,5.332905


EQUIVALENCIA SUPERADA: perdidas y log P dentro de 1e-5; predicciones de caracter identicas. No se desactiva determinismo para forzar el resultado.


...

NVIDIA-SMI: Pliegue 1, hasta seis procesos inicio
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 20 %, 640 MiB, 40960 MiB

Trabajo terminado: fold_1_class_13-17 OK
Trabajo terminado: fold_1_class_18-24 OK
Trabajo terminado: fold_1_class_25-34 OK
NVIDIA-SMI: Pliegue 1, hasta seis procesos mitad de trabajos
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 92 %, 4461 MiB, 40960 MiB

Trabajo terminado: fold_1_class_35-49 OK
Trabajo terminado: fold_1_class_50-64 OK
Trabajo terminado: fold_1_class_65-xx OK


,ok,id,fingerprint,class_index,seconds,epochs,best_epoch,peak_memory_bytes,reused_checkpoint,numeric_policy,device,gpu_name,pid,fresh_training,process_wall_s
0,True,fold_1_class_13-17,22a4313cf0660d5fca23b2a79c5c95f94dec860dd928f2...,0,138.883005,30,30,362789376,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8271,True,141.621604
1,True,fold_1_class_18-24,395eabaf0b771c5f83245269f1adc6dd33151a41c47cca...,1,139.805421,30,30,364119552,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8272,True,142.624524
2,True,fold_1_class_25-34,f95c6e8c039b4805db98ea742f0d7fd6bb1d3066c0cd48...,2,144.364724,30,30,366444544,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8273,True,147.064185
3,True,fold_1_class_35-49,b5ca62a30373308ecd005c8f4aed3c6da33b12da4bb155...,3,151.712588,30,30,368847872,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8274,True,154.515322
4,True,fold_1_class_50-64,34a346c4347c6c997480cf5cb6afa02cebb45fc73a8301...,4,157.095085,30,30,371790336,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8275,True,159.820374
5,True,fold_1_class_65-xx,b35f9a697f37cbcd94caf848f981d3ce01cb082ade4a6e...,5,157.363602,30,30,370728960,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8276,True,160.089098


Aceleracion real medida: 1.1244642761678387 | N efectivo: 6


,modo,trabajo,epocas,mejor_epoca,tiempo_proceso_s,tiempo_trabajo_s,memoria_max_MiB,tiempo_total_s,fresco
0,N=1,fold_1_class_13-17,30,30,27.752527,25.186050,345.982910,180.047079,True
1,N=1,fold_1_class_18-24,30,30,27.049785,24.510106,347.251465,180.047079,True
2,N=1,fold_1_class_25-34,30,30,28.900191,26.310831,349.468750,180.047079,True
3,N=1,fold_1_class_35-49,30,30,30.861190,28.310211,351.760742,180.047079,True
4,N=1,fold_1_class_50-64,30,30,32.351565,29.805635,354.566895,180.047079,True
5,N=1,fold_1_class_65-xx,30,30,33.103813,30.547095,353.554688,180.047079,True
6,N=6,fold_1_class_13-17,30,30,141.621604,138.883005,345.982910,160.118096,True
7,N=6,fold_1_class_18-24,30,30,142.624524,139.805421,347.251465,160.118096,True
8,N=6,fold_1_class_25-34,30,30,147.064185,144.364724,349.468750,160.118096,True
9,N=6,fold_1_class_35-49,30,30,154.515322,151.712588,351.760742,160.118096,True


Memoria max: torch.cuda.max_memory_allocated por proceso; no se interpreta la suma de picos como un pico global simultaneo. nvidia-smi muestra el estado global al inicio y a mitad.
PUERTA DE EJECUCION SUPERADA; la CV puede comenzar con N efectivo 6


## Cinco pliegues y predicciones OOF en Drive
Los pliegues son los de v6 reconstruidos sobre su desarrollo. Se cargan predicciones existentes solo si los hashes e indices coinciden. Cada LM usa exclusivamente su clase y el early stopping interno. La referencia KN y v9 se puntuan en las mismas filas exteriores; no se utiliza el F1 exterior para elegir epocas.

In [13]:
assert PARALLEL_READY,'Falta superar equivalencia y benchmark; no se ejecuta ni se recupera la CV antes de esa verificacion.'
oof_v6=np.full(len(train),-1,dtype=np.int64)
oof_v9=np.full(len(train),-1,dtype=np.int64)
fold_results=[]
for fold_number,(fit_idx,valid_idx) in enumerate(folds,1):
    stage=f'fold_{fold_number}'
    signature=fingerprint('fold_predictions',fold=fold_number,fit=index_digest(fit_idx),valid=index_digest(valid_idx))
    path=RUN_DIR/f'{stage}_oof.npz'
    cached=load_npz_checked(path,signature)
    if cached is None:
        started=time.perf_counter()
        pred_v6=predict_kn(fit_idx,valid_idx,stage)
        pred_v9=predict_neural(fit_idx,train.normalized.iloc[valid_idx].tolist(),stage)
        save_npz_checked(path,fingerprint=np.array(signature),indices=valid_idx,pred_v6=pred_v6,pred_v9=pred_v9,seconds=np.array(time.perf_counter()-started))
    else:
        assert np.array_equal(cached['indices'],valid_idx)
        pred_v6,pred_v9=cached['pred_v6'],cached['pred_v9']
        print('OOF recuperado sin reentrenar:',path.name)
    oof_v6[valid_idx]=pred_v6;oof_v9[valid_idx]=pred_v9
    f1_v6=f1_score(y_all[valid_idx],pred_v6,labels=np.arange(6),average='macro',zero_division=0)
    f1_v9=f1_score(y_all[valid_idx],pred_v9,labels=np.arange(6),average='macro',zero_division=0)
    fold_results.append(dict(pliegue=fold_number,f1_v6=f1_v6,f1_v9=f1_v9,diferencia=f1_v9-f1_v6))
    display(pd.DataFrame(fold_results))
assert (oof_v6[idx_development]>=0).all() and (oof_v9[idx_development]>=0).all()
cv_table=pd.DataFrame(fold_results)
difference_mean=float(cv_table.diferencia.mean())
difference_std=float(cv_table.diferencia.std(ddof=1))
oof_f1_v6=f1_score(y_all[idx_development],oof_v6[idx_development],labels=np.arange(6),average='macro',zero_division=0)
oof_f1_v9=f1_score(y_all[idx_development],oof_v9[idx_development],labels=np.arange(6),average='macro',zero_division=0)
save_npz_checked(RUN_DIR/'oof_complete.npz',fingerprint=np.array(RUN_FINGERPRINT),indices=idx_development,ids=train.id.iloc[idx_development].to_numpy(dtype=str),y=y_all[idx_development],pred_v6=oof_v6[idx_development],pred_v9=oof_v9[idx_development])
display(cv_table)
print('OOF macro v6/v9:',oof_f1_v6,oof_f1_v9)
print('Diferencias pareadas: media',difference_mean,'desviacion muestral',difference_std)

NVIDIA-SMI: fold_1 inicio
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 0 %, 640 MiB, 40960 MiB

Trabajo terminado: fold_1_class_18-24 OK
Trabajo terminado: fold_1_class_65-xx OK
Trabajo terminado: fold_1_class_50-64 OK
NVIDIA-SMI: fold_1 mitad de trabajos
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 0 %, 640 MiB, 40960 MiB

Trabajo terminado: fold_1_class_13-17 OK
Trabajo terminado: fold_1_class_25-34 OK
Trabajo terminado: fold_1_class_35-49 OK


,ok,id,fingerprint,class_index,seconds,epochs,best_epoch,peak_memory_bytes,reused_checkpoint,numeric_policy,device,gpu_name,pid,fresh_training,process_wall_s
0,True,fold_1_class_18-24,395eabaf0b771c5f83245269f1adc6dd33151a41c47cca...,1,139.805421,30,30,364119552,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8272,False,2.373889
1,True,fold_1_class_65-xx,b35f9a697f37cbcd94caf848f981d3ce01cb082ade4a6e...,5,157.363602,30,30,370728960,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8276,False,2.404990
2,True,fold_1_class_50-64,34a346c4347c6c997480cf5cb6afa02cebb45fc73a8301...,4,157.095085,30,30,371790336,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8275,False,2.405630
3,True,fold_1_class_13-17,22a4313cf0660d5fca23b2a79c5c95f94dec860dd928f2...,0,138.883005,30,30,362789376,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8271,False,2.407468
4,True,fold_1_class_25-34,f95c6e8c039b4805db98ea742f0d7fd6bb1d3066c0cd48...,2,144.364724,30,30,366444544,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8273,False,2.412469
5,True,fold_1_class_35-49,b5ca62a30373308ecd005c8f4aed3c6da33b12da4bb155...,3,151.712588,30,30,368847872,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,8274,False,2.413906


,pliegue,f1_v6,f1_v9,diferencia
0,1,0.443844,0.371922,-0.071922


Codificacion y padding realizados una vez: fold_2_encoded_batches.pt
NVIDIA-SMI: fold_2 inicio
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 0 %, 640 MiB, 40960 MiB

Trabajo terminado: fold_2_class_13-17 OK
Trabajo terminado: fold_2_class_18-24 OK
Trabajo terminado: fold_2_class_25-34 OK
NVIDIA-SMI: fold_2 mitad de trabajos
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 91 %, 4461 MiB, 40960 MiB

Trabajo terminado: fold_2_class_35-49 OK
Trabajo terminado: fold_2_class_50-64 OK
Trabajo terminado: fold_2_class_65-xx OK


,ok,id,fingerprint,class_index,seconds,epochs,best_epoch,peak_memory_bytes,reused_checkpoint,numeric_policy,device,gpu_name,pid,fresh_training,process_wall_s
0,True,fold_2_class_13-17,6beeec34d15e1a53765c7ff9e177d9b4f6dcbe2d2a65f2...,0,137.054677,30,30,364139008,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,9569,True,139.838371
1,True,fold_2_class_18-24,953c0c2cea9d3d63b24780590c8e6b51b90fbe9f9923fd...,1,137.307085,30,30,364134912,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,9570,True,140.129476
2,True,fold_2_class_25-34,05a78fa71aa6fea7bf0349f76483433e138bc439629434...,2,146.642226,30,30,366690304,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,9571,True,149.495765
3,True,fold_2_class_35-49,1a60804481b21ceaceddc7dc0686e8d14f3932be3c206e...,3,154.402110,30,30,369125376,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,9572,True,157.186213
4,True,fold_2_class_50-64,fe9e28a7c58b6bdfb7fdb6d4f88d9c28970e8745e16ea4...,4,157.644881,30,30,371940352,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,9573,True,160.428120
5,True,fold_2_class_65-xx,36c94bcea274a823a0797d0f583717faff8895cce563c7...,5,158.600144,30,30,372364800,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,9574,True,161.342596


...

Codificacion y padding realizados una vez: fold_5_encoded_batches.pt
NVIDIA-SMI: fold_5 inicio
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 0 %, 640 MiB, 40960 MiB

Trabajo terminado: fold_5_class_18-24 OK
Trabajo terminado: fold_5_class_13-17 OK
Trabajo terminado: fold_5_class_25-34 OK
NVIDIA-SMI: fold_5 mitad de trabajos
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA A100-SXM4-40GB, 91 %, 4431 MiB, 40960 MiB

Trabajo terminado: fold_5_class_35-49 OK
Trabajo terminado: fold_5_class_50-64 OK
Trabajo terminado: fold_5_class_65-xx OK


,ok,id,fingerprint,class_index,seconds,epochs,best_epoch,peak_memory_bytes,reused_checkpoint,numeric_policy,device,gpu_name,pid,fresh_training,process_wall_s
0,True,fold_5_class_18-24,3bb3456bb30bddb267e8894cbaa13df64df163154bf38c...,1,135.967175,30,30,364689920,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,12693,True,138.779289
1,True,fold_5_class_13-17,c1ad731bd460b5fe307f3f17dba6f3c983b250986121f2...,0,140.040510,30,30,364061184,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,12692,True,142.815417
2,True,fold_5_class_25-34,692a9dc4870ff4bd42ea50f6a5a68af6534f2e72866a52...,2,142.566903,30,30,367655424,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,12694,True,145.314803
3,True,fold_5_class_35-49,80ebbe831023e69a1270d9925505c7bf99496f7bf6026c...,3,152.989367,30,30,368498176,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,12695,True,155.824242
4,True,fold_5_class_50-64,4d3fbd7a177fa470a55e2f1a4750aa4391bb82ddc31c06...,4,157.200763,30,30,371959296,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,12697,True,159.926384
5,True,fold_5_class_65-xx,962b8373e19b74b01aa813d714186aebd8d00c069329db...,5,157.711047,30,30,371885056,False,"{'determinista': True, 'precision': 'fp32', 't...",cuda:0,NVIDIA A100-SXM4-40GB,12696,True,160.451572


,pliegue,f1_v6,f1_v9,diferencia
0,1,0.443844,0.371922,-0.071922
1,2,0.444670,0.377804,-0.066865
2,3,0.433306,0.376171,-0.057134
3,4,0.442226,0.372701,-0.069524
4,5,0.441174,0.374878,-0.066296


,pliegue,f1_v6,f1_v9,diferencia
0,1,0.443844,0.371922,-0.071922
1,2,0.444670,0.377804,-0.066865
2,3,0.433306,0.376171,-0.057134
3,4,0.442226,0.372701,-0.069524
4,5,0.441174,0.374878,-0.066296


OOF macro v6/v9: 0.44105568409189616 0.3751575207315581
Diferencias pareadas: media -0.0663484562333116 desviacion muestral 0.005619615891343988


## Regla de decision fijada
Solo se avanza si la media de las diferencias es positiva y supera su desviacion muestral. No se calibran umbrales o sesgos. Esta regla es una heuristica, no una prueba formal de significacion.

In [14]:
PASA=bool(difference_mean>0 and difference_mean>difference_std)
print('PASA' if PASA else 'NO PASA')
if not PASA:print('Holdout y submission omitidos: la regla pareada no se cumple.')


NO PASA
Holdout y submission omitidos: la regla pareada no se cumple.


## Holdout unico y persistido
Solo si PASA se ajustan seis LSTM con el mismo protocolo interno sobre desarrollo y se evalua el holdout. Un resultado ya guardado se recupera sin volver a predecir ni recalcular metricas. No se cambian parametros tras verlo.

In [15]:
if PASA:
    assert 'holdout_f1' not in globals(),'El holdout ya se ha consultado en este kernel.'
    signature=fingerprint('holdout',fit=index_digest(idx_development),valid=index_digest(idx_holdout))
    metrics_path=RUN_DIR/'holdout_metrics.json'
    pred_path=RUN_DIR/'holdout_predictions.npz'
    if metrics_path.exists():
        with metrics_path.open(encoding='utf-8') as handle:holdout_metrics=json.load(handle)
        assert holdout_metrics['fingerprint']==signature
        cached=load_npz_checked(pred_path,signature)
        assert cached is not None and np.array_equal(cached['indices'],idx_holdout)
        holdout_pred=cached['predictions']
        print('Holdout recuperado: NO se vuelve a evaluar.')
    else:
        holdout_pred=predict_neural(idx_development,train.normalized.iloc[idx_holdout].tolist(),'holdout')
        value=f1_score(y_all[idx_holdout],holdout_pred,labels=np.arange(6),average='macro',zero_division=0)
        report=classification_report(y_all[idx_holdout],holdout_pred,labels=np.arange(6),target_names=CLASSES,output_dict=True,zero_division=0)
        matrix=confusion_matrix(y_all[idx_holdout],holdout_pred,labels=np.arange(6))
        holdout_metrics=dict(fingerprint=signature,f1_macro=float(value),report=report,matrix=matrix.tolist())
        save_npz_checked(pred_path,fingerprint=np.array(signature),indices=idx_holdout,predictions=holdout_pred)
        with metrics_path.open('x',encoding='utf-8') as handle:json.dump(holdout_metrics,handle,ensure_ascii=False,indent=2)
    holdout_f1=float(holdout_metrics['f1_macro'])
    print('F1 macro holdout v9:',holdout_f1)
    display(pd.DataFrame(holdout_metrics['report']).T.loc[CLASSES,['precision','recall','f1-score','support']])
    display(pd.DataFrame(holdout_metrics['matrix'],index=CLASSES,columns=CLASSES))
else:
    print('HOLDOUT OMITIDO: NO PASA; no se entrena ni se evalua el holdout.')


HOLDOUT OMITIDO: NO PASA; no se entrena ni se evalua el holdout.


## Ajuste final y submission condicional
Si PASA, se aplica el mismo procedimiento de ajuste/early stopping interno a todo el train suministrado. Los modelos finales y las predicciones quedan en ARTIFACT_DIR/RUN_ID. El test solo se utiliza para predecir; `age_submission.csv` determina cabecera y orden de IDs. La submission final se guarda exclusivamente en SUBMISSION_PATH, fuera de la carpeta del experimento; si ya existe un CSV distinto, se detiene sin sobrescribirlo.

In [16]:
if PASA:
    final_signature=fingerprint('final_predictions',train=TRAIN_SHA,test=hashlib.sha256(TEST_PATH.read_bytes()).hexdigest())
    final_path=RUN_DIR/'final_predictions.npz'
    cached=load_npz_checked(final_path,final_signature)
    test_clean=test_raw.text.fillna('').str.strip().map(html.unescape).map(normalize_text).tolist()
    if cached is None:
        final_predictions=predict_neural(np.arange(len(train)),test_clean,'final')
        save_npz_checked(final_path,fingerprint=np.array(final_signature),predictions=final_predictions,ids=test_raw.id.to_numpy(dtype=str))
    else:
        assert np.array_equal(cached['ids'],test_raw.id.to_numpy(dtype=str))
        final_predictions=cached['predictions']
    assert len(final_predictions)==len(test_raw)
    assert set(sample.id)==set(test_raw.id) and len(sample)==len(test_raw),'Los IDs de sample y test no coinciden; detener.'
    predicted=pd.DataFrame({'id':test_raw.id,'age_range':CLASSES[final_predictions]}).set_index('id')
    submission=predicted.loc[sample.id].reset_index()
    submission=submission.rename(columns={canonical:actual for canonical,actual in sample_mapping.items()})[sample_raw.columns]
    assert len(submission)==len(test_raw) and submission[sample_mapping['id']].is_unique
    assert submission[sample_mapping['age_range']].isin(CLASSES).all()
    payload=submission.to_csv(index=False,sep=',').encode('utf-8')
    submission_path=SUBMISSION_PATH
    submission_path.parent.mkdir(parents=True,exist_ok=True)
    if submission_path.exists():assert submission_path.read_bytes()==payload,'Ya existe una submission distinta; no sobrescribir.'
    else:
        with submission_path.open('xb') as handle:handle.write(payload)
    print('Submission lista:',submission_path,'filas',len(submission),'cabecera',submission.columns.tolist())
    display(submission.head())
else:
    print('SUBMISSION OMITIDA: NO PASA. No se ajustan modelos finales ni se genera CSV.')

SUBMISSION OMITIDA: NO PASA. No se ajustan modelos finales ni se genera CSV.


## Tabla final y limites de la comparacion
La tabla pareada es la evidencia principal; el score publico de Kaggle no es una medicion comparable con holdout. Las semillas y checkpoints permiten repetir/reanudar, pero no garantizan identidad bit a bit entre distintas versiones de CUDA o hardware.

In [17]:
display(cv_table)
summary=pd.DataFrame([dict(modelo='v6 KN recalculado',f1_oof=oof_f1_v6),dict(modelo='v9 LSTM one-hot',f1_oof=oof_f1_v9)])
display(summary)
print('Media de diferencias:',difference_mean,'DE pareada ddof=1:',difference_std,'Decision:','PASA' if PASA else 'NO PASA')
print('Hardware medido:',BENCHMARK_REPORT['gpu'],'| DETERMINISTA:',BENCHMARK_REPORT['determinista'],'| precision:',BENCHMARK_REPORT['precision'],'| TF32:',BENCHMARK_REPORT['tf32'])
print('Prueba pliegue 1: secuencial',BENCHMARK_REPORT['sequential_seconds'],'s; paralela',BENCHMARK_REPORT['parallel_seconds'],'s; aceleracion',BENCHMARK_REPORT['speedup'],'x; concurrencia efectiva',BENCHMARK_REPORT['effective_parallelism'])
print('Medicion fresca comparable:',BENCHMARK_REPORT['fresh_measurement'],'; una medicion con caches no habilita paralelismo mayor que 1.')
display(pd.DataFrame(BENCHMARK_REPORT['table']))
if EXECUTION_REPORTS:
    runtime_table=pd.DataFrame([dict(trabajo=row['id'],epocas=row['epochs'],mejor_epoca=row['best_epoch'],tiempo_proceso_s=row['process_wall_s'],tiempo_trabajo_historico_s=row['seconds'],memoria_max_MiB=row['peak_memory_bytes']/1048576,reutilizado=not row.get('fresh_training',False)) for row in EXECUTION_REPORTS])
    display(runtime_table)
    print('Memoria maxima registrada por un trabajo de esta ejecucion:',runtime_table.memoria_max_MiB.max(),'MiB. No es la suma ni un pico global simultaneo.')
if PASA:
    print('Holdout v9:',holdout_f1)
    if SUBMISSION_PATH.is_file():print('Submission:',SUBMISSION_PATH)
    else:print('No existe la submission en el destino configurado:',SUBMISSION_PATH)
else:
    print('Holdout no consultado; submission no generada.')
print('Pesos, scores por clase, OOF, metricas, logs y benchmark guardados en Drive:',RUN_DIR)
print('No se envio nada a Kaggle. No se modificaron archivos de v6.')

,pliegue,f1_v6,f1_v9,diferencia
0,1,0.443844,0.371922,-0.071922
1,2,0.444670,0.377804,-0.066865
2,3,0.433306,0.376171,-0.057134
3,4,0.442226,0.372701,-0.069524
4,5,0.441174,0.374878,-0.066296


,modelo,f1_oof
0,v6 KN recalculado,0.441056
1,v9 LSTM one-hot,0.375158


Media de diferencias: -0.0663484562333116 DE pareada ddof=1: 0.005619615891343988 Decision: NO PASA
Hardware medido: NVIDIA A100-SXM4-40GB | DETERMINISTA: True | precision: fp32 | TF32: False
Prueba pliegue 1: secuencial 180.047079209 s; paralela 160.11809625700016 s; aceleracion 1.1244642761678387 x; concurrencia efectiva 6
Medicion fresca comparable: True ; una medicion con caches no habilita paralelismo mayor que 1.


,modo,trabajo,epocas,mejor_epoca,tiempo_proceso_s,tiempo_trabajo_s,memoria_max_MiB,tiempo_total_s,fresco
0,N=1,fold_1_class_13-17,30,30,27.752527,25.186050,345.982910,180.047079,True
1,N=1,fold_1_class_18-24,30,30,27.049785,24.510106,347.251465,180.047079,True
2,N=1,fold_1_class_25-34,30,30,28.900191,26.310831,349.468750,180.047079,True
3,N=1,fold_1_class_35-49,30,30,30.861190,28.310211,351.760742,180.047079,True
4,N=1,fold_1_class_50-64,30,30,32.351565,29.805635,354.566895,180.047079,True
5,N=1,fold_1_class_65-xx,30,30,33.103813,30.547095,353.554688,180.047079,True
6,N=6,fold_1_class_13-17,30,30,141.621604,138.883005,345.982910,160.118096,True
7,N=6,fold_1_class_18-24,30,30,142.624524,139.805421,347.251465,160.118096,True
8,N=6,fold_1_class_25-34,30,30,147.064185,144.364724,349.468750,160.118096,True
9,N=6,fold_1_class_35-49,30,30,154.515322,151.712588,351.760742,160.118096,True


,trabajo,epocas,mejor_epoca,tiempo_proceso_s,tiempo_trabajo_historico_s,memoria_max_MiB,reutilizado
0,fold_1_class_18-24,30,30,2.373889,139.805421,347.251465,True
1,fold_1_class_65-xx,30,30,2.404990,157.363602,353.554688,True
2,fold_1_class_50-64,30,30,2.405630,157.095085,354.566895,True
3,fold_1_class_13-17,30,30,2.407468,138.883005,345.982910,True
4,fold_1_class_25-34,30,30,2.412469,144.364724,349.468750,True
5,fold_1_class_35-49,30,30,2.413906,151.712588,351.760742,True
6,fold_2_class_13-17,30,30,139.838371,137.054677,347.270020,False
7,fold_2_class_18-24,30,30,140.129476,137.307085,347.266113,False
8,fold_2_class_25-34,30,30,149.495765,146.642226,349.703125,False
9,fold_2_class_35-49,30,30,157.186213,154.402110,352.025391,False


Memoria maxima registrada por un trabajo de esta ejecucion: 355.84375 MiB. No es la suma ni un pico global simultaneo.
Holdout no consultado; submission no generada.
Pesos, scores por clase, OOF, metricas, logs y benchmark guardados en Drive: /content/drive/MyDrive/PLN/RESULTS/tweetage_v9/v9_lstm_onehot_h256_c140_seed42_v1
No se envio nada a Kaggle. No se modificaron archivos de v6.
